# Pandas Avanzado

Este notebook cubre operaciones avanzadas en pandas para análisis y transformación de datos.


## Objetivos de este notebook

En este notebook veremos:

1. `assign`, `pipe` y encadenamiento de operaciones
2. `query` y `eval`
3. Ordenamiento avanzado y ranking
4. Funciones ventana con `transform`
5. `map`, `replace`, `where` y `mask`
6. Manejo avanzado de fechas
7. Strings con `.str`
8. `apply` vs alternativas vectorizadas
9. `explode`
10. MultiIndex
11. Agregaciones múltiples
12. Reshape con `melt`, `stack`, `unstack`
13. Joins avanzados: `merge_asof` y `merge_ordered`
14. Validaciones y calidad de datos
15. Ejercicio integrador


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 120)

## 1. Creamos una base de ejemplo

In [2]:
ventas = pd.DataFrame({
    'cliente_id': [101, 101, 102, 102, 103, 103, 104, 104],
    'ciudad': ['Bogotá', 'Bogotá', 'Medellín', 'Medellín', 'Cali', 'Cali', 'Barranquilla', 'Barranquilla'],
    'segmento': ['Pyme', 'Pyme', 'Corp', 'Corp', 'Pyme', 'Pyme', 'Corp', 'Corp'],
    'fecha': pd.to_datetime(['2025-01-05', '2025-02-10', '2025-01-15', '2025-03-20', '2025-01-08', '2025-02-25', '2025-01-10', '2025-03-01']),
    'producto': ['A', 'B', 'A', 'C', 'B', 'C', 'A', 'B'],
    'monto': [1200, 1800, 2500, 3200, 900, 1500, 2700, 2100],
    'unidades': [3, 4, 5, 6, 2, 3, 5, 4],
    'canal': ['Web', 'Oficina', 'Web', 'Aliado', 'Web', 'Oficina', 'Aliado', 'Web'],
    'tags': [['nuevo', 'promo'], ['upsell'], ['vip'], ['vip', 'renovacion'], ['promo'], ['nuevo'], ['vip'], ['promo', 'crosssell']]
})

ventas

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]"
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell]
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip]
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]"
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo]
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo]
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip]
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]"


## 2. Encadenamiento de operaciones con `assign` y `pipe`

Esto ayuda a escribir transformaciones más limpias y legibles.

In [3]:
resultado = (
    ventas
    .assign(
        precio_unitario=lambda df: df['monto'] / df['unidades'],
        mes=lambda df: df['fecha'].dt.to_period('M').astype(str)
    )
)

resultado

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,precio_unitario,mes
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",400.000000,2025-01
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],450.000000,2025-02
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],500.000000,2025-01
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",533.333333,2025-03
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],450.000000,2025-01
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],500.000000,2025-02
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],540.000000,2025-01
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",525.000000,2025-03


### Ejemplo con `pipe`

In [4]:
def filtrar_pymes(df):
    return df[df['segmento'] == 'Pyme']

def agregar_ticket_promedio(df):
    return df.assign(ticket_promedio=df['monto'] / df['unidades'])

ventas.pipe(filtrar_pymes).pipe(agregar_ticket_promedio)

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,ticket_promedio
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",400.0
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],450.0
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],450.0
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],500.0


## 3. `query` y `eval`

Sirven para escribir filtros y expresiones de manera más compacta.

In [5]:
ventas.query("segmento == 'Corp' and monto > 2500")

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]"
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip]


In [6]:
ventas.eval("ingreso_por_unidad = monto / unidades")

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,ingreso_por_unidad
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",400.000000
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],450.000000
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],500.000000
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",533.333333
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],450.000000
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],500.000000
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],540.000000
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",525.000000


## 4. Ordenamiento avanzado y ranking

In [7]:
ventas.sort_values(['segmento', 'monto'], ascending=[True, False])

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]"
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip]
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip]
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]"
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell]
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo]
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]"
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo]


In [8]:
ventas.assign(
    rank_monto_segmento=ventas.groupby('segmento')['monto'].rank(method='dense', ascending=False)
)

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,rank_monto_segmento
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",3.0
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],1.0
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],3.0
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",1.0
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],4.0
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],2.0
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],2.0
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",4.0


## 5. `transform`: métricas por grupo sin perder el detalle

Muy útil cuando quieres calcular una métrica agregada y mantener cada fila original.

In [9]:
ventas.assign(
    promedio_segmento=ventas.groupby('segmento')['monto'].transform('mean'),
    share_vs_segmento=ventas['monto'] / ventas.groupby('segmento')['monto'].transform('sum')
)

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,promedio_segmento,share_vs_segmento
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",1350.0,0.222222
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],1350.0,0.333333
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],2625.0,0.238095
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",2625.0,0.304762
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],1350.0,0.166667
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],1350.0,0.277778
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],2625.0,0.257143
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",2625.0,0.200000


## 6. `map`, `replace`, `where` y `mask`

In [10]:
mapa_segmento = {'Pyme': 'Pequeña/Mediana', 'Corp': 'Corporativo'}

ventas['segmento'].map(mapa_segmento)

0    Pequeña/Mediana
1    Pequeña/Mediana
2        Corporativo
3        Corporativo
4    Pequeña/Mediana
5    Pequeña/Mediana
6        Corporativo
7        Corporativo
Name: segmento, dtype: str

In [11]:
ventas['canal'].replace({'Web': 'Digital', 'Oficina': 'Presencial', 'Aliado': 'Partners'})

0       Digital
1    Presencial
2       Digital
3      Partners
4       Digital
5    Presencial
6      Partners
7       Digital
Name: canal, dtype: str

In [12]:
ventas.assign(
    monto_capado=ventas['monto'].where(ventas['monto'] <= 2500, 2500),
    monto_solo_altos=ventas['monto'].mask(ventas['monto'] < 2000, np.nan)
)

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,monto_capado,monto_solo_altos
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",1200,NaN
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],1800,NaN
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],2500,2500.0
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",2500,3200.0
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],900,NaN
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],1500,NaN
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],2500,2700.0
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",2100,2100.0


## 7. Fechas: componentes, offsets y resample

In [13]:
ventas.assign(
    anio=ventas['fecha'].dt.year,
    mes=ventas['fecha'].dt.month,
    dia_semana=ventas['fecha'].dt.day_name()
)

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,anio,mes,dia_semana
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",2025,1,Sunday
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],2025,2,Monday
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],2025,1,Wednesday
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",2025,3,Thursday
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],2025,1,Wednesday
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],2025,2,Tuesday
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],2025,1,Friday
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",2025,3,Saturday


## 8. Operaciones con strings usando `.str`

In [15]:
clientes = pd.DataFrame({
    'nombre': [' Ana Pérez ', 'juan gomez', 'MARIA LOPEZ'],
    'email': ['ana@test.com', 'juan@test.com', 'maria@test.com']
})

clientes.assign(
    nombre_limpio=clientes['nombre'].str.strip().str.title(),
    dominio=clientes['email'].str.split('@').str[1]
)

,nombre,email,nombre_limpio,dominio
0,Ana Pérez,ana@test.com,Ana Pérez,test.com
1,juan gomez,juan@test.com,Juan Gomez,test.com
2,MARIA LOPEZ,maria@test.com,Maria Lopez,test.com


## 9. `apply` vs alternativas vectorizadas

Regla práctica: si puedes resolverlo con operaciones vectorizadas, mejor que con `apply`.

In [16]:
# Menos eficiente
ventas['tipo_monto_apply'] = ventas['monto'].apply(lambda x: 'alto' if x >= 2000 else 'bajo')

# Más idiomático y escalable
ventas['tipo_monto_npwhere'] = np.where(ventas['monto'] >= 2000, 'alto', 'bajo')

ventas[['monto', 'tipo_monto_apply', 'tipo_monto_npwhere']]

,monto,tipo_monto_apply,tipo_monto_npwhere
0,1200,bajo,bajo
1,1800,bajo,bajo
2,2500,alto,alto
3,3200,alto,alto
4,900,bajo,bajo
5,1500,bajo,bajo
6,2700,alto,alto
7,2100,alto,alto


## 10. Expandir listas con `explode`

In [17]:
ventas[['cliente_id', 'producto', 'tags']].explode('tags')

,cliente_id,producto,tags
0,101,A,nuevo
0,101,A,promo
1,101,B,upsell
2,102,A,vip
3,102,C,vip
3,102,C,renovacion
4,103,B,promo
5,103,C,nuevo
6,104,A,vip
7,104,B,promo


## 11. MultiIndex

Útil cuando trabajas con varias dimensiones jerárquicas.

In [18]:
multi = ventas.groupby(['segmento', 'canal'])['monto'].agg(['sum', 'mean', 'count'])
multi

sum    mean  count
segmento canal                       
Corp     Aliado   5900  2950.0      2
         Web      4600  2300.0      2
Pyme     Oficina  3300  1650.0      2
         Web      2100  1050.0      2

In [19]:
multi.loc[('Corp', 'Aliado')]

sum      5900.0
mean     2950.0
count       2.0
Name: (Corp, Aliado), dtype: float64

## 12. Agregaciones múltiples con `agg`

In [20]:
ventas.groupby('segmento').agg(
    monto_total=('monto', 'sum'),
    monto_promedio=('monto', 'mean'),
    unidades_max=('unidades', 'max'),
    n_clientes=('cliente_id', 'nunique')
)

,monto_total,monto_promedio,unidades_max,n_clientes
segmento,,,,
Corp,10500,2625.0,6,2
Pyme,5400,1350.0,4,2


## 13. Reshape con `melt`, `stack` y `unstack`

In [21]:
resumen = ventas.groupby('segmento')[['monto', 'unidades']].sum().reset_index()
resumen

,segmento,monto,unidades
0,Corp,10500,20
1,Pyme,5400,12


In [22]:
resumen_largo = resumen.melt(id_vars='segmento', var_name='metrica', value_name='valor')
resumen_largo

,segmento,metrica,valor
0,Corp,monto,10500
1,Pyme,monto,5400
2,Corp,unidades,20
3,Pyme,unidades,12


In [23]:
tabla = ventas.groupby(['segmento', 'canal'])['monto'].sum()
tabla

segmento  canal  
Corp      Aliado     5900
          Web        4600
Pyme      Oficina    3300
          Web        2100
Name: monto, dtype: int64

In [24]:
tabla.unstack(fill_value=0)

canal,Aliado,Oficina,Web
segmento,,,
Corp,5900,0,4600
Pyme,0,3300,2100


## 14. Ejercicio integrador


In [ ]:
ventas = pd.DataFrame({
    'cliente_id': [101, 101, 102, 102, 103, 103, 104, 104],
    'ciudad': ['Bogotá', 'Bogotá', 'Medellín', 'Medellín', 'Cali', 'Cali', 'Barranquilla', 'Barranquilla'],
    'segmento': ['Pyme', 'Pyme', 'Corp', 'Corp', 'Pyme', 'Pyme', 'Corp', 'Corp'],
    'fecha': pd.to_datetime(['2025-01-05', '2025-02-10', '2025-01-15', '2025-03-20', '2025-01-08', '2025-02-25', '2025-01-10', '2025-03-01']),
    'producto': ['A', 'B', 'A', 'C', 'B', 'C', 'A', 'B'],
    'monto': [1200, 1800, 2500, 3200, 900, 1500, 2700, 2100],
    'unidades': [3, 4, 5, 6, 2, 3, 5, 4],
    'canal': ['Web', 'Oficina', 'Web', 'Aliado', 'Web', 'Oficina', 'Aliado', 'Web'],
    'tags': [['nuevo', 'promo'], ['upsell'], ['vip'], ['vip', 'renovacion'], ['promo'], ['nuevo'], ['vip'], ['promo', 'crosssell']]
})

ventas

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]"
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell]
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip]
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]"
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo]
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo]
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip]
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]"


Objetivo:

1. Crear `precio_unitario`
2. Crear `mes`
3. Calcular el ranking de `monto` dentro de cada `segmento`
4. Calcular el share del `monto` frente al total del segmento
5. Explotar la columna `tags`
6. Construir una tabla final por `segmento`, `mes` y `tag`

In [29]:
ejercicio = ventas.copy()

ejercicio["precio_unitario"] = ejercicio["monto"] / ejercicio["unidades"]
ejercicio["mes"] = ejercicio["fecha"].dt.to_period("M").astype(str)
ejercicio["rank_monto"] = ejercicio.groupby("segmento")["monto"].rank(method="dense", ascending=False)
ejercicio["share_segmento"] = ejercicio["monto"] / ejercicio.groupby("segmento")["monto"].transform("sum")


In [30]:
ejercicio

,cliente_id,ciudad,segmento,fecha,producto,monto,unidades,canal,tags,precio_unitario,mes,rank_monto,share_segmento
0,101,Bogotá,Pyme,2025-01-05,A,1200,3,Web,"[nuevo, promo]",400.000000,2025-01,3.0,0.222222
1,101,Bogotá,Pyme,2025-02-10,B,1800,4,Oficina,[upsell],450.000000,2025-02,1.0,0.333333
2,102,Medellín,Corp,2025-01-15,A,2500,5,Web,[vip],500.000000,2025-01,3.0,0.238095
3,102,Medellín,Corp,2025-03-20,C,3200,6,Aliado,"[vip, renovacion]",533.333333,2025-03,1.0,0.304762
4,103,Cali,Pyme,2025-01-08,B,900,2,Web,[promo],450.000000,2025-01,4.0,0.166667
5,103,Cali,Pyme,2025-02-25,C,1500,3,Oficina,[nuevo],500.000000,2025-02,2.0,0.277778
6,104,Barranquilla,Corp,2025-01-10,A,2700,5,Aliado,[vip],540.000000,2025-01,2.0,0.257143
7,104,Barranquilla,Corp,2025-03-01,B,2100,4,Web,"[promo, crosssell]",525.000000,2025-03,4.0,0.200000


In [31]:
ejercicio = (
    ejercicio
    .explode("tags")
    .groupby(["segmento", "mes", "tags"], as_index=False)
    .agg(
        monto_total=("monto", "sum"),
        clientes=("cliente_id", "nunique"),
        ticket_promedio=("precio_unitario", "mean")
    )
)

ejercicio

,segmento,mes,tags,monto_total,clientes,ticket_promedio
0,Corp,2025-01,vip,5200,2,520.000000
1,Corp,2025-03,crosssell,2100,1,525.000000
2,Corp,2025-03,promo,2100,1,525.000000
3,Corp,2025-03,renovacion,3200,1,533.333333
4,Corp,2025-03,vip,3200,1,533.333333
5,Pyme,2025-01,nuevo,1200,1,400.000000
6,Pyme,2025-01,promo,2100,2,425.000000
7,Pyme,2025-02,nuevo,1500,1,500.000000
8,Pyme,2025-02,upsell,1800,1,450.000000


In [32]:
ejercicio = (
    ventas
    .assign(
        precio_unitario=lambda df: df['monto'] / df['unidades'],
        mes=lambda df: df['fecha'].dt.to_period('M').astype(str),
        rank_monto=lambda df: df.groupby('segmento')['monto'].rank(method='dense', ascending=False),
        share_segmento=lambda df: df['monto'] / df.groupby('segmento')['monto'].transform('sum')
    )
    .explode('tags')
    .groupby(['segmento', 'mes', 'tags'], as_index=False)
    .agg(
        monto_total=('monto', 'sum'),
        clientes=('cliente_id', 'nunique'),
        ticket_promedio=('precio_unitario', 'mean')
    )
)

ejercicio

,segmento,mes,tags,monto_total,clientes,ticket_promedio
0,Corp,2025-01,vip,5200,2,520.000000
1,Corp,2025-03,crosssell,2100,1,525.000000
2,Corp,2025-03,promo,2100,1,525.000000
3,Corp,2025-03,renovacion,3200,1,533.333333
4,Corp,2025-03,vip,3200,1,533.333333
5,Pyme,2025-01,nuevo,1200,1,400.000000
6,Pyme,2025-01,promo,2100,2,425.000000
7,Pyme,2025-02,nuevo,1500,1,500.000000
8,Pyme,2025-02,upsell,1800,1,450.000000
